# defect-inspector: training

Choose `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

If the session stops, press `Run all` again. Training continues from the last checkpoint in Google
Drive. Fill in the fields below for each object; use a new `RUN_ID` to start over.

In [ ]:
OBJECT_ID = "metal-nut"  #@param {type:"string"}
OBJECT_NAME = "Metal nut"  #@param {type:"string"}
DATASET_ARCHIVE = "metal_nut.tar.xz"  #@param {type:"string"}
RUN_ID = "run-1"  #@param {type:"string"}
TRAIN_STEPS = 70000  #@param {type:"integer"}
IMAGE_SIZE = 256  #@param {type:"integer"}
USE_AUTOENCODER = True  #@param {type:"boolean"}

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/defect-inspector")
for name in ("datasets", "penalty", "runs"):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)

In [ ]:
import os

REPO = Path("/content/defect-inspector")
if REPO.exists():
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/tudorraducristian/defect-inspector.git {REPO}
!pip install -q onnx==1.23.2 onnxruntime==1.31.1
os.environ["PYTHONPATH"] = "{}:{}".format(REPO / "src", REPO / "training")

In [ ]:
import urllib.request

COMMIT = "fcab5146f84ae17597044ad5ddf1656ccf805401"
TEACHER = Path("/content/teacher_small.pth")
TEACHER_BYTES = 10779715
if not TEACHER.exists():
    url = "https://raw.githubusercontent.com/nelson1425/EfficientAD/{}/models/teacher_small.pth".format(COMMIT)
    urllib.request.urlretrieve(url, TEACHER)
if TEACHER.stat().st_size != TEACHER_BYTES:
    TEACHER.unlink()
    raise RuntimeError("The teacher file has the wrong size; run this cell again.")

In [ ]:
import shutil

archive = DRIVE_ROOT / "datasets" / DATASET_ARCHIVE
if not archive.exists():
    raise FileNotFoundError("Put {} in Google Drive, in defect-inspector/datasets/.".format(DATASET_ARCHIVE))

# Reading from the local disk, because every step would otherwise wait for Drive.
extracted = Path("/content/data") / OBJECT_ID
if not extracted.exists():
    shutil.unpack_archive(str(archive), str(extracted))
found = list(extracted.glob("**/train/good"))
if len(found) != 1:
    raise RuntimeError("Expected one train/good folder in the archive, found {}.".format(len(found)))
DATASET_ROOT = found[0].parents[1]
print(DATASET_ROOT)

In [ ]:
PENALTY_ARCHIVE = DRIVE_ROOT / "penalty" / "imagenette2-320.tgz"
if not PENALTY_ARCHIVE.exists():
    partial = PENALTY_ARCHIVE.with_name(PENALTY_ARCHIVE.name + ".part")
    urllib.request.urlretrieve("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz", partial)
    os.replace(partial, PENALTY_ARCHIVE)

PENALTY_ROOT = Path("/content/penalty")
if not PENALTY_ROOT.exists():
    shutil.unpack_archive(str(PENALTY_ARCHIVE), str(PENALTY_ROOT))

In [ ]:
RUN_DIR = DRIVE_ROOT / "runs" / OBJECT_ID / RUN_ID
autoencoder_flag = "" if USE_AUTOENCODER else "--no-autoencoder"
station = DATASET_ROOT / "station.json"
station_flag = '--station "{}"'.format(station) if station.exists() else ""
!cd {REPO} && python -m defect_train.cli --require-gpu --dataset-root "{DATASET_ROOT}" --object-id "{OBJECT_ID}" --object-name "{OBJECT_NAME}" --teacher "{TEACHER}" --penalty "{PENALTY_ROOT}" --out "{RUN_DIR}" --train-steps {TRAIN_STEPS} --image-size {IMAGE_SIZE} {autoencoder_flag} {station_flag}
if _exit_code != 0:
    raise RuntimeError("Training stopped; see the message above.")

In [ ]:
print((RUN_DIR / "metrics.json").read_text())
print("Download this folder from Drive:", RUN_DIR)